<a href="https://colab.research.google.com/github/LindsayRendon/campaigns-demo/blob/master/05_NLP_DistilBERT_IMDB_Sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. NLP con Transformers para análisis de sentimiento

## 2. Objetivo
Hacer un "Fine-tuning" (ajuste fino) de un modelo Transformer preentrenado (DistilBERT) para clasificar reseñas de películas como positivas o negativas.

## 3. Qué problema resuelve
Las empresas reciben miles de comentarios diarios (redes sociales, Amazon, encuestas). Leerlos a mano para saber si los clientes están felices o enojados es imposible. Este modelo resuelve el problema del **Análisis de Sentimiento a gran escala**, extrayendo la intención detrás del texto de forma automatizada y con comprensión profunda del contexto.

## 4. Dataset usado
**IMDB Reviews**. Un dataset clásico con 50,000 reseñas de películas en inglés (25k para entrenamiento, 25k para prueba). Las reseñas están fuertemente polarizadas: o son muy buenas (etiqueta 1) o muy malas (etiqueta 0).

---
### Conceptos teóricos clave aplicados en este Notebook:

* **¿Qué es NLP (Natural Language Processing)?** Es la rama de la IA que permite a las computadoras entender, interpretar y generar lenguaje humano de forma útil.
* **¿Qué es tokenización?** Las redes neuronales no entienden letras. La tokenización es el proceso de trocear una frase en pedazos ("tokens", que pueden ser palabras o sílabas) y asignarles un número de identificación (ID) único de un diccionario.
* **¿Qué son los embeddings?** Son vectores (listas de números) que representan el *significado* de un token. En un espacio de embeddings, las palabras con significados similares (ej. "rey" y "reina") están matemáticamente cerca.
* **¿Qué es un Transformer?** Es una arquitectura de red neuronal introducida por Google en 2017. A diferencia de las redes recurrentes (RNN) que leen palabra por palabra, el Transformer lee toda la frase a la vez y usa "Auto-atención" para descubrir qué palabras están relacionadas entre sí, sin importar qué tan lejos estén en la frase.
* **¿Qué es BERT / DistilBERT?** BERT es un modelo base creado por Google preentrenado leyendo toda la Wikipedia. **DistilBERT** es su "hermano menor": una versión destilada (comprimida) que es un 40% más pequeña y un 60% más rápida, pero retiene el 97% de la inteligencia del original.
* **Bag of Words vs Transformer:** *Bag of Words* (Bolsa de palabras) cuenta cuántas veces aparece una palabra, pero destruye el orden. Para BoW, "No es bueno, es malo" y "No es malo, es bueno" son matemáticamente idénticas. El *Transformer* sí entiende el orden y el contexto.
* **¿Qué es Fine-tuning?** Es tomar un modelo masivo que ya sabe la gramática y el idioma en general (como DistilBERT), y entrenarlo un poco más (añadiéndole una capa final) con tus datos específicos (ej. IMDB) para que se vuelva experto en esa tarea concreta.

In [1]:
# Instalamos las librerías necesarias de Hugging Face
# Los Transformers necesitan un par de librerías modernas de la empresa Hugging Face. Colab no las trae todas por defecto, así que las instalamos primero
!pip install -q transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00


In [2]:
# 5. Librerías (Vamos a cargar las librerías)
import numpy as np
import pandas as pd
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import matplotlib.pyplot as plt

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


In [4]:
# 6. Carga y exploración de datos
# Cargamos los datos y seleccionamos una pequeña muestra. Si entrenamos con los 50,000 textos tardaría horas en Colab. Con 2000 textos será suficiente para ver lo listo que es el modelo.
# Descargamos el dataset IMDB directamente desde Hugging Face
dataset = load_dataset("stanfordnlp/imdb")

# Para que el ejercicio se ejecute rápido en Colab, tomaremos una muestra pequeña:
# 2000 reseñas para entrenar y 500 para evaluar.
small_train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
small_test_dataset = dataset["test"].shuffle(seed=42).select(range(500))

print("Ejemplo de reseña:")
print("TEXTO:", small_train_dataset[0]['text'][:300], "...")
print("ETIQUETA:", "Positiva (1)" if small_train_dataset[0]['label'] == 1 else "Negativa (0)")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Ejemplo de reseña:
TEXTO: There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier's plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot  ...
ETIQUETA: Positiva (1)


In [5]:
# 7. Preprocesamiento: Tokenización
# La tokenización: pasamos las palabras a números usando el diccionario oficial de DistilBERT.
# Cargamos el tokenizador del modelo que vamos a usar
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Creamos una función que tokenizará los textos, cortándolos si son muy largos
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

# Aplicamos la tokenización a nuestros sets de datos
tokenized_train = small_train_dataset.map(tokenize_function, batched=True)
tokenized_test = small_test_dataset.map(tokenize_function, batched=True)

print("Datos tokenizados y listos para la red neuronal.")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Datos tokenizados y listos para la red neuronal.


In [7]:
# 8. Modelo y 9. Entrenamiento
# Cargamos el modelo, definimos cómo medir el éxito (Accuracy) e iniciamos el entrenamiento.
# Cargamos el modelo pre-entrenado, avisándole que nuestra salida tiene 2 etiquetas (Positivo/Negativo)
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# Función para calcular la precisión (Accuracy)
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Configuración del entrenamiento
training_args = TrainingArguments(
    output_dir="test_trainer",
    eval_strategy="epoch",  # <-- ¡AQUÍ ESTÁ EL CAMBIO! (antes decía evaluation_strategy)
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

# Inicializamos el "Entrenador" (Trainer) de Hugging Face
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

# ¡A entrenar!
print("Iniciando Fine-Tuning. Esto tomará unos 2-3 minutos con GPU...")
trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Iniciando Fine-Tuning. Esto tomará unos 2-3 minutos con GPU...


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.295477,0.876000
2,No log,0.339111,0.874000
3,No log,0.318471,0.890000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=375, training_loss=0.27995890299479165, metrics={'train_runtime': 341.4026, 'train_samples_per_second': 17.575, 'train_steps_per_second': 1.098, 'total_flos': 794804391936000.0, 'train_loss': 0.27995890299479165, 'epoch': 3.0})

In [8]:
# 10. Evaluación final y análisis de predicciones
# Vamos a revisar el Accuracy y buscar ejemplos donde acertó y donde falló.
print("\\nEvaluación final en el set de Test:")
resultados = trainer.evaluate()
print(f"Accuracy final: {resultados['eval_accuracy'] * 100:.2f}%")

# Vamos a sacar las predicciones reales sobre el set de prueba
predicciones_crudas = trainer.predict(tokenized_test)
predicciones_clases = np.argmax(predicciones_crudas.predictions, axis=-1)

# Creamos un DataFrame para ver mejor los datos
df_resultados = pd.DataFrame({
    'Texto': small_test_dataset['text'],
    'Real': small_test_dataset['label'],
    'Prediccion': predicciones_clases
})

# Mapeamos 0 y 1 a texto para leerlo más fácil
mapa_labels = {0: 'Negativo', 1: 'Positivo'}
df_resultados['Real_Text'] = df_resultados['Real'].map(mapa_labels)
df_resultados['Pred_Text'] = df_resultados['Prediccion'].map(mapa_labels)

# Separamos aciertos y errores
aciertos = df_resultados[df_resultados['Real'] == df_resultados['Prediccion']]
errores = df_resultados[df_resultados['Real'] != df_resultados['Prediccion']]

\nEvaluación final en el set de Test:


Training Loss,Validation Loss,Epoch,Accuracy
No log,0.318471,3,0.890000


Accuracy final: 89.00%


In [9]:
# Mostrar algunos aciertos (Ejemplos de frases positivas y negativas)
# Mostramos los ejemplos
print("="*60)
print("✅ EJEMPLOS DE PREDICCIONES CORRECTAS:")
print("="*60)

# Una positiva correcta
ej_pos = aciertos[aciertos['Real'] == 1].iloc[0]
print(f"[REAL: {ej_pos['Real_Text']} | PREDICCIÓN: {ej_pos['Pred_Text']}]")
print(f"Texto: {ej_pos['Texto'][:400]}...\\n")

# Una negativa correcta
ej_neg = aciertos[aciertos['Real'] == 0].iloc[0]
print(f"[REAL: {ej_neg['Real_Text']} | PREDICCIÓN: {ej_neg['Pred_Text']}]")
print(f"Texto: {ej_neg['Texto'][:400]}...\\n")

print("="*60)
print("❌ EJEMPLOS DE ERRORES DEL MODELO:")
print("="*60)

if len(errores) > 0:
    ej_err = errores.iloc[0]
    print(f"[REAL: {ej_err['Real_Text']} | PREDICCIÓN: {ej_err['Pred_Text']}]")
    print(f"Texto: {ej_err['Texto'][:500]}...")
    print("-> (A menudo falla con el sarcasmo, reseñas que empiezan bien y acaban mal, o ironías).")
else:
    print("¡El modelo no cometió errores en esta muestra!")

✅ EJEMPLOS DE PREDICCIONES CORRECTAS:
[REAL: Positivo | PREDICCIÓN: Positivo]
Texto: <br /><br />When I unsuspectedly rented A Thousand Acres, I thought I was in for an entertaining King Lear story and of course Michelle Pfeiffer was in it, so what could go wrong?<br /><br />Very quickly, however, I realized that this story was about A Thousand Other Things besides just Acres. I started crying and couldn't stop until long after the movie ended. Thank you Jane, Laura and Jocelyn, f...\n
[REAL: Negativo | PREDICCIÓN: Negativo]
Texto: This movie was so frustrating. Everything seemed energetic and I was totally prepared to have a good time. I at least thought I'd be able to stand it. But, I was wrong. First, the weird looping? It was like watching "America's Funniest Home Videos". The damn parents. I hated them so much. The stereo-typical Latino family? I need to speak with the person responsible for this. We need to have a talk...\n
❌ EJEMPLOS DE ERRORES DEL MODELO:
[REAL: Negativo | PRED

## 11. Conclusión
Con apenas 3 épocas y utilizando solo 2000 reseñas (una fracción minúscula del dataset original), hemos logrado hacer un Fine-Tuning de DistilBERT alcanzando un Accuracy superior al 85% (dependiendo de la semilla de inicialización). El ecosistema de Transformers de Hugging Face (`Trainer`, `AutoModel`) encapsula toda la complejidad matemática de la auto-atención, permitiéndonos adaptar un modelo colosal a una necesidad de negocio concreta en muy pocas líneas de código.

## 12. Qué aprendí
* **El inmenso poder del Transfer Learning:** Comprobé por qué ya casi nadie entrena redes recurrentes (RNN o LSTM) desde cero para NLP. DistilBERT ya traía el idioma "aprendido" en sus Embeddings. Solo tuvimos que actualizar un poco sus pesos (con un *learning rate* muy bajo de `2e-5`) para alinear su conocimiento hacia la polaridad de las películas.
* **Manejo de Contexto y Sarcasmo:** A diferencia del enfoque clásico de *Bag of Words*, al revisar los errores me doy cuenta de que el modelo intenta entender la frase entera. Cuando falla, suele ser en reseñas extremadamente largas que contienen tanto elementos muy positivos como quejas al final, confundiendo al clasificador en su intento de ponderar el sentimiento global.
* **Ventaja de la arquitectura Transformer:** Al ver que el modelo procesaba lotes enteros de texto sin bucles secuenciales, pude ver la ventaja de paralelización masiva de los Transformers, lo que hace posible entrenarlos en GPUs de forma muchísimo más eficiente que las arquitecturas secuenciales antiguas.